## Rewoo Agent
Here we can use the Python SDK to develop the simple rewoo agent, then save the agent to a config.yaml and run it from there.

In [1]:
import os
import sys

# Import the NeMo-Agent-Toolkit module
module_path = os.path.abspath('../../../src/')
if module_path not in sys.path:
    sys.path.insert(0, module_path)

In [2]:
import logging

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)

In [ ]:
from nat.agent.rewoo_agent.register import ReWOOAgentWorkflow
from nat.llm.nim_llm import NimLLM
from nat.plugins.langchain.tools.tavily_internet_search import TavilyInternetSearchTool
from nat.utils.sdk.nat_workflow import NatWorkflow
from nat_multi_frameworks.haystack_agent import HaystackChitchatTool
from nat_simple_calculator.register import CalculatorToolGroup

llm = NimLLM(
    model_name="nvdev/meta/llama-3.3-70b-instruct",
    temperature=0,
    max_tokens=4096,
    name="nim_llm",
)

haystack_llm = NimLLM(
    model_name="meta/llama-3.1-405b-instruct",
    temperature=0.2,
    max_tokens=1024,
    name="haystack_llm",
)

calculator_tool_group = CalculatorToolGroup(
    name="calculator",
)

internet_search_tool = TavilyInternetSearchTool(
    name="internet_search"
)

chitchat_agent_tool = HaystackChitchatTool(
    llm=haystack_llm,
    name="haystack_chitchat_agent"
)

agent = ReWOOAgentWorkflow(
    tools=[internet_search_tool, chitchat_agent_tool, calculator_tool_group],
    llm=llm,
    verbose=True,
    tool_call_max_retries=3,
)

nat_workflow = NatWorkflow(
    entrypoint=agent,
)

/Users/spastoriza/Documents/Programming/public/nat-fork/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
await nat_workflow.prompt("Who was Djikstra?")

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


'Edsger Dijkstra, a Dutch computer scientist born on May 11, 1930, in Rotterdam, Netherlands, and died on August 6, 2002, in Nuenen, Netherlands, who developed the paradigm of structured programming and made fundamental contributions to computer science.'

In [ ]:
import os
from pathlib import Path

path_to_yaml = Path(os.getcwd(), "config", "config.yaml").resolve()

# Create the config directory if it doesn't exist
if not path_to_yaml.parent.exists():
    os.makedirs(path_to_yaml.parent)

# Save the workflow to a config file
nat_workflow.save_to_config_file(path_to_yaml)

# Print out the config file content
with open(path_to_yaml) as f:
    print(f.read())

functions:
  internet_search:
    _type: tavily_internet_search
  haystack_chitchat_agent:
    _type: haystack_chitchat_agent
    llm_name: meta/llama-3.1-405b-instruct

function_groups:
  calculator:
    _type: calculator

llms:
  nim_llm:
    _type: nim
    model: nvdev/meta/llama-3.3-70b-instruct
    max_tokens: 4096
    temperature: 0.0

workflow:
  _type: rewoo_agent
  llm_name: nim_llm
  verbose: true
  tool_names:
  - internet_search
  - haystack_chitchat_agent
  - calculator
  tool_call_max_retries: 3



In [ ]:
from pathlib import Path

from nat.eval.rag_evaluator.register import RagasEvaluator
from nat.utils.sdk.nat_evaluation import EvalDatasetJsonConfig
from nat.utils.sdk.nat_evaluation import NatEvaluation

path_to_dataset = Path(os.path.curdir, "../../../../", "examples/agents/data/rewoo.json").resolve()

accuracy_evaluator = RagasEvaluator(
    llm=llm,
    metric="AnswerAccuracy",
    name="accuracy"
)

relevance_evaluator = RagasEvaluator(
    llm=llm,
    metric="ContextRelevance",
    name="relevance"
)

response_groundedness_evaluator = RagasEvaluator(
    llm=llm,
    metric="ResponseGroundedness",
    name="groundedness"
)

evaluation = NatEvaluation(
    output_dir=Path(".tmp/nat/examples/rewoo_agent/"),
    dataset=EvalDatasetJsonConfig(file_path=path_to_dataset),
    evaluators=[accuracy_evaluator, relevance_evaluator, response_groundedness_evaluator],
)

nat_workflow.add_evaluator(evaluation)

In [ ]:
path_to_yaml = Path(os.getcwd(), "config", "eval_config.yaml").resolve()

# Create the config directory if it doesn't exist
if not path_to_yaml.parent.exists():
    os.makedirs(path_to_yaml.parent)

# Save the workflow to a config file
nat_workflow.save_to_config_file(path_to_yaml)

# Print out the config file content
with open(path_to_yaml) as f:
    print(f.read())

functions:
  internet_search:
    _type: tavily_internet_search
  haystack_chitchat_agent:
    _type: haystack_chitchat_agent
    llm_name: meta/llama-3.1-405b-instruct

function_groups:
  calculator:
    _type: calculator

llms:
  nim_llm:
    _type: nim
    model: nvdev/meta/llama-3.3-70b-instruct
    max_tokens: 4096
    temperature: 0.0

workflow:
  _type: rewoo_agent
  llm_name: nim_llm
  verbose: true
  tool_names:
  - internet_search
  - haystack_chitchat_agent
  - calculator
  tool_call_max_retries: 3

eval:
  general:
    max_concurrency: 8
    workflow_alias: null
    output_dir: .tmp/nat/examples/rewoo_agent
    output: null
    dataset:
      _type: json
      id_key: id
      structure:
        disable: false
        question_key: question
        answer_key: answer
        generated_answer_key: generated_answer
        trajectory_key: intermediate_steps
        expected_trajectory_key: expected_intermediate_steps
      filter:
        allowlist: null
        denylist: nul

In [ ]:
await nat_workflow.evaluate()

Evaluating Ragas nv_accuracy:   0%|          | 0/5 [00:00<?, ?it/s]



Evaluating Ragas nv_response_groundedness: 100%|██████████| 5/5 [00:01<00:00,  3.36it/s]

Evaluating Ragas nv_accuracy: 100%|██████████| 5/5 [00:03<00:00,  1.31it/s]

